In [10]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import settings
from src.aggregations import territorial, person, labor,cadunico, general
import pandas as pd 
import numpy as np
from pathlib import Path
import importlib

importlib.reload(settings)
importlib.reload(territorial)
importlib.reload(person)
importlib.reload(labor)
importlib.reload(cadunico)
importlib.reload(general)

pd.set_option("display.float_format", "{:,.3f}".format)
pd.set_option("display.max_columns", None)

In [ ]:
df_cubo_old = pd.read_excel(settings.CUBO_PATH)

In [2]:
df_cubo = pd.read_excel(settings.DATA_PATH /'input_data' /'dados_cubo_final_v0__2026-05-28_16-48.xlsx')

In [8]:
df_aux = pd.read_parquet(settings.DATA_PATH / 'input_data'/ 'non-public' / 'df_aux_cubo.parquet')
df_aux['chave'] = df_aux["ente_bbagil"] + "-" + df_aux["documento_beneficiario_bbagil"]


# Seção 1

In [ ]:
# [Big Number] O ticket médio geral do Ciclo 1 da Política Nacional Aldir Blanc foi de R$ 12,8 mil. 
df_valor_geral = general.calcular_ticket_medio_df_aux(df_aux=df_aux)

In [ ]:
# Section 1
# bignumber1.csv

# Valor executado por estado x municipio abs / percentual 
df_cubo_est = df_cubo[df_cubo['tipo_ente'] == 'ESTADO']
df_cubo_mun = df_cubo[df_cubo['tipo_ente'] == 'MUNICIPIO']

# Totais executados com ceil
valor_estados = np.ceil(df_cubo_est["valor_transacao"].sum())
valor_municipios = np.ceil(df_cubo_mun["valor_transacao"].sum())

# Valores de referência
total_estados = 1_510_000_000
total_municipios = 1_490_000_000

df_execucao = pd.DataFrame({
    "Estados_DF": [valor_estados],
    "Municipios_DF": [valor_municipios],
    "perc_executado_estados": [valor_estados / total_estados],
    "perc_executado_municipios": [valor_municipios / total_municipios],
})

df_execucao
df_execucao.to_csv('s1_bn1.csv')

,Estados_DF,Municipios_DF,perc_executado_estados,perc_executado_municipios
0,"1,450,514,327.000","1,395,481,268.000",0.961,0.937


In [3]:
# executed_value_by.csv
df_states       = territorial.executed_value_n_contemplados_qty_by(df_cubo=df_cubo, by_filter='ESTADO')
df_municipality = territorial.executed_value_n_contemplados_qty_by(df_cubo=df_cubo, by_filter='MUNICIPIO')
df_uf           = territorial.executed_value_n_contemplados_qty_by(df_cubo=df_cubo, by_filter='UF')

# df_states.to_csv(settings.DATA_PATH_SECTION1 / 'executed_value_by_state.csv')
# df_municipality.to_csv(settings.DATA_PATH_SECTION1 / 'executed_value_by_municipality.csv')
# df_uf.to_csv(settings.DATA_PATH_SECTION1 / 'executed_value_by_uf.csv')

In [38]:
# resumo_valores_uf_estado.csv
df_uf_cut = territorial.criar_df_uf_cut_from_aux(df_aux=df_aux, by_filter='UF')
df_uf_cut.to_csv(settings.DATA_PATH_SECTION2 / 'resumo_valores_uf_uf.csv')

In [36]:
df_uf_cut

,uf,macrorregiao,visao,valor_executado_rs,valor_executado_perc,valor_executado_perc_regiao,min_valor,mediana_valor,max_valor,media_valor,media_aparada_1pct_valor,valor_executado_percapita,qtde_contemplados,perc_qtde_contemplados_regiao
0,GO,Centro-Oeste,ESTADO,"55,985,604.870",0.039,0.447,"10,000.000","80,000.000","2,000,000.000","118,865.403","110,707.933",7.617,471,0.361
1,MT,Centro-Oeste,ESTADO,"24,571,646.220",0.017,0.196,"25,000.000","73,000.000","1,800,000.000","102,810.235","86,243.557",6.405,239,0.183
2,MS,Centro-Oeste,ESTADO,"23,316,653.570",0.016,0.186,"2,000.000","15,000.000","2,000,000.000","44,839.718","34,581.636",8.035,520,0.398
3,DF,Centro-Oeste,ESTADO,"21,309,243.880",0.015,0.170,"12,575.090","100,000.000","2,000,000.000","280,384.788","257,456.585",7.144,76,0.058
4,CE,Nordeste,ESTADO,"78,727,917.790",0.054,0.176,"2,769.160","49,000.000","3,228,537.250","86,229.921","72,105.361",8.526,913,0.091
5,BA,Nordeste,ESTADO,"76,394,220.840",0.053,0.170,"3,438.450","40,000.000","2,886,960.000","64,304.900","58,354.873",5.144,1188,0.118
6,PE,Nordeste,ESTADO,"75,277,246.240",0.052,0.168,965.980,"25,000.000","2,356,525.570","33,590.918","29,856.284",7.891,2241,0.223
7,MA,Nordeste,ESTADO,"63,390,851.190",0.044,0.141,"2,651.910","30,000.000","7,792,386.600","80,241.584","59,097.473",9.042,790,0.079
8,PB,Nordeste,ESTADO,"37,177,439.730",0.026,0.083,"1,000.000","10,000.000","1,300,000.000","21,183.726","18,333.325",8.969,1755,0.175
9,AL,Nordeste,ESTADO,"35,128,148.390",0.024,0.078,"3,000.000","15,000.000","2,634,948.390","27,681.756","22,807.086",10.909,1269,0.126


In [31]:
# aggregate_faixa_valor_ju_wide_by_uf.csv
df_uf_new = territorial.aggregate_faixa_valor_ju_wide_by_uf(df_cubo=df_cubo)
# df_uf_new.to_csv(settings.DATA_PATH_SECTION2 / 'aggregate_faixa_valor_ju_wide_by_uf.csv')

In [32]:
df_uf_new

,uf,total_contemplados_uf,valor_total_uf,qtd_ate_2_mil,perc_qtd_ate_2_mil,valor_ate_2_mil,perc_valor_ate_2_mil,qtd_de_2_a_10_mil,perc_qtd_de_2_a_10_mil,valor_de_2_a_10_mil,perc_valor_de_2_a_10_mil,qtd_de_10_a_50_mil,perc_qtd_de_10_a_50_mil,valor_de_10_a_50_mil,perc_valor_de_10_a_50_mil,qtd_de_50_a_200_mil,perc_qtd_de_50_a_200_mil,valor_de_50_a_200_mil,perc_valor_de_50_a_200_mil,qtd_acima_de_200_mil,perc_qtd_acima_de_200_mil,valor_acima_de_200_mil,perc_valor_acima_de_200_mil
0,AC,1079,"24,329,783.680",86,0.080,"119,410.020",0.005,404,0.374,"2,713,109.200",0.112,539,0.500,"12,812,365.550",0.527,40,0.037,"3,525,137.400",0.145,10,0.009,"5,159,761.510",0.212
1,AL,6378,"60,294,994.130",2543,0.399,"3,045,933.360",0.051,2566,0.402,"12,627,446.230",0.209,1121,0.176,"25,239,552.430",0.419,129,0.020,"11,384,127.010",0.189,19,0.003,"7,997,935.100",0.133
2,AM,2075,"74,122,348.910",253,0.122,"388,629.700",0.005,575,0.277,"3,147,992.640",0.042,929,0.448,"27,779,370.170",0.375,289,0.139,"28,336,046.250",0.382,29,0.014,"14,470,310.150",0.195
3,AP,649,"23,633,686.160",181,0.279,"222,658.850",0.009,216,0.333,"1,381,748.300",0.058,187,0.288,"4,462,053.260",0.189,40,0.062,"3,970,638.100",0.168,25,0.039,"13,596,587.650",0.575
4,BA,17500,"176,909,114.840",5776,0.330,"7,798,862.070",0.044,8920,0.510,"42,307,782.740",0.239,2294,0.131,"58,629,214.980",0.331,438,0.025,"42,126,855.730",0.238,72,0.004,"26,046,399.320",0.147
5,CE,8256,"136,622,936.070",2739,0.332,"3,723,850.040",0.027,3561,0.431,"18,032,030.320",0.132,1478,0.179,"37,819,829.870",0.277,408,0.049,"43,411,131.190",0.318,70,0.008,"33,636,094.650",0.246
6,DF,393,"41,170,700.290",98,0.249,"133,862.920",0.003,91,0.232,"373,251.580",0.009,39,0.099,"1,071,727.780",0.026,136,0.346,"13,086,856.380",0.318,29,0.074,"26,505,001.630",0.644
7,ES,1883,"55,730,593.340",35,0.019,"47,574.130",0.001,916,0.486,"5,622,769.830",0.101,696,0.370,"17,359,185.210",0.311,210,0.112,"19,527,596.510",0.350,26,0.014,"13,173,467.660",0.236
8,GO,4320,"105,504,440.400",954,0.221,"1,218,562.190",0.012,2000,0.463,"10,967,483.440",0.104,890,0.206,"24,427,850.350",0.232,398,0.092,"40,947,519.240",0.388,78,0.018,"27,943,025.180",0.265
9,MA,8366,"114,921,030.050",3406,0.407,"4,624,102.760",0.040,3381,0.404,"16,070,620.220",0.140,1201,0.144,"28,781,224.320",0.250,308,0.037,"30,079,637.510",0.262,70,0.008,"35,365,445.240",0.308


In [49]:
# aggregate_faixa_valor_ju_wide_by_state.csv
df_uf_new = territorial.aggregate_faixa_valor_ju_wide_by_uf(df_cubo=df_cubo, by_filter='ESTADO')
df_uf_new.to_csv(settings.DATA_PATH_SECTION2 / 'aggregate_faixa_valor_ju_wide_by_state.csv')

In [19]:
# executed_value_by_region
df_states_region       = territorial.aggregate_execution_by_region(df_cubo=df_cubo, by_filter='ESTADO')
df_municipality_region = territorial.aggregate_execution_by_region(df_cubo=df_cubo, by_filter='MUNICIPIO')
df_uf_region           = territorial.aggregate_execution_by_region(df_cubo=df_cubo, by_filter='UF')

# df_states_region.to_csv(settings.DATA_PATH_SECTION1 / 'executed_value_by_region_state.csv')
# df_municipality_region.to_csv(settings.DATA_PATH_SECTION1 / 'executed_value_by_region_municipality.csv')
# df_uf_region.to_csv(settings.DATA_PATH_SECTION1 / 'executed_value_by_region_uf.csv')


In [20]:
df_uf_region

,regiao,valor_executado_rs,qtde_contemplados,min_valor,mediana_valor,max_valor,media_valor,populacao,perc_valor_executado,perc_qtde_contemplados,perc_populacao,perc_contemplados_populacao,qtd_tipo_documento_CNPJ,qtd_tipo_documento_CPF,valor_tipo_documento_CNPJ,valor_tipo_documento_CPF,min_valor_tipo_documento_CNPJ,min_valor_tipo_documento_CPF,mediana_valor_tipo_documento_CNPJ,mediana_valor_tipo_documento_CPF,max_valor_tipo_documento_CNPJ,max_valor_tipo_documento_CPF,media_valor_tipo_documento_CNPJ,media_valor_tipo_documento_CPF
0,Centro-Oeste,"237,420,986.840",9335,378.000,"5,804.235","2,000,000.000","25,784.208",17071595,0.083,0.056,0.080,0.001,1898,7437,"116,959,750.110","120,461,236.730",386.700,378.000,"15,570.000","5,000.000","2,000,000.000","735,000.000","62,983.172","16,387.054"
1,Nordeste,"852,268,894.220",79446,380.000,"3,000.000","7,792,386.600","11,226.177",57112096,0.299,0.476,0.269,0.001,9063,70383,"393,254,434.490","459,014,459.730",380.000,380.000,"14,800.000","2,500.000","7,792,386.600","440,000.000","46,129.552","6,811.011"
2,Norte,"303,553,050.960",14504,400.000,"6,025.000","22,109,764.920","21,631.373",18669345,0.107,0.087,0.088,0.001,1576,12928,"153,453,413.560","150,099,637.400",450.000,400.000,"29,934.280","5,000.000","22,109,764.920","280,000.000","100,956.193","11,995.496"
3,Sudeste,"1,066,203,024.520",45655,375.000,"6,904.320","8,000,000.000","23,836.419",88617693,0.375,0.274,0.417,0.001,11917,33738,"668,453,933.040","397,749,091.480",375.000,375.000,"15,950.000","5,000.000","8,000,000.000","600,000.000","57,990.278","11,979.312"
4,Sul,"386,549,637.450",17946,375.000,"8,802.500","2,400,000.000","22,040.691",31113021,0.136,0.108,0.146,0.001,7826,10120,"259,190,161.370","127,359,476.080",375.000,380.000,"12,727.790","6,877.010","2,400,000.000","461,000.000","34,567.906","12,685.207"


In [51]:
# aggregate_values_by
df_municipality_agg     = territorial.aggregate_execution_summary_by_scope(df_cubo=df_cubo, scope='MUNICIPIO')
df_state_agg            = territorial.aggregate_execution_summary_by_scope(df_cubo=df_cubo, scope='ESTADO')
df_capital_agg          = territorial.aggregate_capital_interior_summary(df_cubo=df_cubo)

# df_municipality_agg.to_csv(settings.DATA_PATH_SECTION1 / 'aggregate_values_by_municipality.csv')
# df_capital_agg.to_csv(settings.DATA_PATH_SECTION1 / 'aggregate_values_by_capital.csv')
# df_state_agg.to_csv(settings.DATA_PATH_SECTION1 / 'aggregate_values_by_state.csv')

In [ ]:
# values_by_population_size
df_population_size = territorial.aggregate_execution_by_porte_with_estado(df_cubo=df_cubo)
denominador_urbano_rural = (
    df_population_size["valor_urbano_por_porte"]
    + df_population_size["valor_rural_por_porte"]
)

df_population_size["percentual_valor_urbano_por_porte"] = np.where(
    denominador_urbano_rural.ne(0),
    df_population_size["valor_urbano_por_porte"] / denominador_urbano_rural,
    np.nan
)

df_population_size["percentual_valor_rural_por_porte"] = np.where(
    denominador_urbano_rural.ne(0),
    df_population_size["valor_rural_por_porte"] / denominador_urbano_rural,
    np.nan
)
# df_population_size.to_csv(settings.DATA_PATH_SECTION1 / 'values_by_population_size.csv')

In [54]:
# resumo_por_porte_populacional versão RESUMIDA - Estamos utilizando esse
df_resumo_por_porte_populacional = territorial.resumo_por_porte_populacional(df_aux=df_aux)
# df_resumo_por_porte_populacional.to_csv(settings.DATA_PATH_SECTION2 / 'resumo_por_porte_populacional.csv')

KeyError: "Label(s) ['chave'] do not exist"

In [24]:
# resumo_valor_por_porte_municipio.csv
df_population_size_mean = territorial.resumo_valor_por_porte_municipio(df_cubo=df_cubo)
# df_population_size_mean.to_csv(settings.DATA_PATH_SECTION1 / 'population_size_mean.csv')

In [4]:
# values_by_special_territory
df_special_territory_municipality = territorial.aggregate_special_territories_by(
    df_cubo=df_cubo, 
    categories=settings.CATEGORIES_SPECIAL_TERRITORIES, 
    by_filter="MUNICIPIO"
)

df_special_territory_state = territorial.aggregate_special_territories_by(
    df_cubo=df_cubo, 
    categories=settings.CATEGORIES_SPECIAL_TERRITORIES, 
    by_filter="ESTADO"
)

df_special_territory_uf = territorial.aggregate_special_territories_by(
    df_cubo=df_cubo, 
    categories=settings.CATEGORIES_SPECIAL_TERRITORIES, 
    by_filter="UF"
)

# df_special_territory_municipality.to_csv(settings.DATA_PATH_SECTION1 / 'values_by_special_territory_municipality.csv')
# df_special_territory_state.to_csv(settings.DATA_PATH_SECTION1 / 'values_by_special_territory_state.csv')
# df_special_territory_uf.to_csv(settings.DATA_PATH_SECTION1 / 'values_by_special_territory_uf.csv')


In [86]:
# special_territory_w_ibge_by_brazil
df_vis_territorio_brasil = territorial.generate_special_territories_brazil_view(df_cubo=df_cubo)
# df_vis_territorio_brasil.to_csv(settings.DATA_PATH_SECTION1 / 'special_territory_w_ibge_by_brazil.csv')

In [88]:
# aggregate_by_local_residencia
df_interior_rm_uf = territorial.aggregate_by_local_residencia(df_cubo=df_cubo, visao='uf')
# df_interior_rm_uf.to_csv(settings.DATA_PATH_SECTION1 / 'aggregate_by_local_residencia_uf.csv')

In [96]:
df_interior_rm_estado = territorial.aggregate_by_local_residencia(df_cubo=df_cubo, visao='estado')

In [111]:
df_interior_por_uf_estado = territorial.aggregate_estado_by_uf_local_residencia(df_cubo=df_cubo)

In [115]:
#aggregate_estado_by_uf_local_residencia
df_interior_por_uf_estado.to_csv(settings.DATA_PATH_SECTION1 / 'aggregate_estado_by_uf_local_residencia.csv')

In [ ]:
# df_interior_rm_estado
df_interior_rm_estado.to_csv(settings.DATA_PATH_SECTION1 / 'aggregate_by_local_residencia_estado.csv')


# Section 2

In [ ]:
#[Frase destaque] Enquanto a maior parte dos agentes culturais contemplados receberam pagamento na faixa de 2 a 10 mil reais, a maior parte do recurso executado foi para pagamentos na faixa de 10 a 50 mil reais. 
#[Gráfico] Gráfico xx - Comparativo entre Distribuição do Nº Contemplados e Distribuição de Recursos Executados (R$), por Faixa de Valor 

df_faixa_valor = territorial.aggregate_faixa_valor_ju_by(df_cubo=df_cubo)
# df_faixa_valor.to_csv(settings.DATA_PATH_SECTION2 / 'values_range_by_brazil_v2.csv')

In [ ]:
# [Texto] O gráfico abaixo apresenta o comparativo entre os contemplados pela Aldir Blanc e a concentração da população em cada região do país;
# [Gráfico]  Gráfico X - Comparativo entre Distribuição dos Agentes Culturais Contemplados e Distribuição da População por Região

df_uf_region           = territorial.aggregate_execution_by_region(df_cubo=df_cubo, by_filter='UF')
# df_uf_region[['regiao', 'perc_qtde_contemplados', 'perc_populacao']]

In [ ]:
# [Texto] A distribuição de recursos nas diferentes faixas de valor permite visualizar quais regiões concentram seus recursos em pagamentos maiores - com maiores percentuais nas maiores faixas - e quais distribuiram os recursos em pagamentos menores - com destaque para a região nordeste. 
# [Gráfico]  Gráfico xx - Distribuição dos Contemplados, por Faixa de Valor Recebido, nas Regiões 

df_agg_faixas_regiao = territorial.aggregate_faixa_valor_ju_wide_by_regiao(df_cubo=df_cubo)
# df_agg_faixas_regiao[['regiao','perc_qtd_ate_2_mil', 'perc_qtd_de_2_a_10_mil', 'perc_qtd_de_10_a_50_mil', 'perc_qtd_de_50_a_200_mil', 'qtd_acima_de_200_mil']]

In [ ]:
# [Texto] A partir da análise por região, percebe-se que a maioria dos contemplados recebeu pagamentos na faixa de 2 a 10 mil reais. Essa tendência é presente em todas as regiões do Brasil, com exceção do Nordeste, que registrou pagamentos de até 2 mil reais para a maioria dos contemplados (41,5%).
# [Texto] A alta presença de pagamentos até 2 mil reais na região Nordeste faz com que a região possua um ticket médio de R$ 8,2 mil, o menor entre todas as regiões.
# [Pendência: Incluir gráfico de mapa coroplético, com o ticket médio por região] 

# resumo_valores_uf_estado.csv
df_uf_cut = territorial.criar_df_uf_cut_from_aux(df_aux=df_aux, by_filter='UF')
df_uf_cut.to_csv(settings.DATA_PATH_SECTION2 / 'resumo_valores_uf.csv')

In [ ]:
# [Texto] A maioria das unidades federativas direcionaram a maior parte do recurso executado para os contemplados que receberam de 10 a 50 mil reais. 
# [Gráfico] Gráfico xx - Distribuição dos Recursos Executados nas Unidades Federativas, por Faixa de Valor 

# aggregate_faixa_valor_ju_wide_by_uf.csv
df_uf_new = territorial.aggregate_faixa_valor_ju_wide_by_uf(df_cubo=df_cubo)
df_uf_new.to_csv(settings.DATA_PATH_SECTION2 / 'aggregate_faixa_valor_ju_wide_by_uf.csv')
#df_uf_new[['uf','perc_valor_ate_2_mil', 'perc_valor_de_2_a_10_mil', 'perc_valor_de_10_a_50_mil', 'perc_valor_de_50_a_200_mil', 'perc_valor_acima_de_200_mil']]

In [54]:
# [Texto] O gráfico a seguir apresenta como os agentes culturais contemplados pelos entes estaduais foram distribuídos nas faixas de valores. Pode-se observar que a maioria recebeu valores entre 10 e 50 mil reais, o que equivale a 51,9%.
# [Gráfico] Gráfico xx - Distribuição dos contemplados, por faixa de valor, nos Estados

# aggregate_faixa_valor_ju_wide_by_state.csv
df_uf_new = territorial.aggregate_faixa_valor_ju_wide_by_uf(df_cubo=df_cubo, by_filter='ESTADO')
# df_uf_new.to_csv(settings.DATA_PATH_SECTION2 / 'aggregate_faixa_valor_ju_wide_by_state.csv')

In [ ]:
#[Gráfico] Gráfico xx - Distribuição dos Contemplados, por Faixa de Valor Recebido, nos Municípios por Porte de Município

# resumo_faixa_valor_por_porte.csv
df_resumo_faixas_porte = territorial.resumo_faixa_valor_por_porte(df_cubo=df_cubo)
# df_resumo_faixas_porte.to_csv(settings.DATA_PATH_SECTION2 / 'faixa_valor_porte_populacional.csv')
# df_resumo_faixas_porte[['porte_populacional', 'perc_qtd_contemplados_ate_2_mil','perc_qtd_contemplados_de_2_a_10_mil','perc_qtd_contemplados_de_10_a_50_mil','perc_qtd_contemplados_de_50_a_200_mil','perc_qtd_contemplados_acima_de_200_mil', ]]

In [ ]:
# [Gráfico] Gráfico xx - Distribuição dos Recursos, por Faixa de Valor, nos Municípios por Porte de Município
# resumo_faixa_valor_por_porte.csv
df_resumo_faixas_porte = territorial.resumo_faixa_valor_por_porte(df_cubo=df_cubo)
# df_resumo_faixas_porte.to_csv(settings.DATA_PATH_SECTION2 / 'faixa_valor_porte_populacional.csv')
# df_resumo_faixas_porte[['porte_populacional', 'perc_valor_contemplados_ate_2_mil','perc_valor_contemplados_de_2_a_10_mil','perc_valor_contemplados_de_10_a_50_mil','perc_valor_contemplados_de_50_a_200_mil','perc_valor_contemplados_acima_de_200_mil', ]]

In [61]:
df_resumo_faixas_porte

,porte_populacional,total_qtd_contemplados,total_valor_transacao,qtd_contemplados_acima_de_200_mil,perc_qtd_contemplados_acima_de_200_mil,valor_transacao_acima_de_200_mil,perc_valor_transacao_acima_de_200_mil,qtd_contemplados_de_10_a_50_mil,perc_qtd_contemplados_de_10_a_50_mil,valor_transacao_de_10_a_50_mil,perc_valor_transacao_de_10_a_50_mil,qtd_contemplados_de_50_a_200_mil,perc_qtd_contemplados_de_50_a_200_mil,valor_transacao_de_50_a_200_mil,perc_valor_transacao_de_50_a_200_mil,qtd_contemplados_de_2_a_10_mil,perc_qtd_contemplados_de_2_a_10_mil,valor_transacao_de_2_a_10_mil,perc_valor_transacao_de_2_a_10_mil,qtd_contemplados_ate_2_mil,perc_qtd_contemplados_ate_2_mil,valor_transacao_ate_2_mil,perc_valor_transacao_ate_2_mil
0,-99,22050,"1,450,514,326.100",1104,0.050,"592,272,160.590",0.408,11454,0.519,"317,106,059.640",0.219,4924,0.223,"509,229,282.040",0.351,4479,0.203,"31,762,705.320",0.022,89,0.004,"144,118.510",0.000
1,1_pequeno_i,57289,"266,445,332.420",7,0.000,"1,796,825.730",0.007,4511,0.079,"90,736,359.520",0.341,447,0.008,"33,425,238.510",0.125,23629,0.412,"106,275,294.550",0.399,28695,0.501,"34,211,614.110",0.128
2,2_pequeno_ii,38944,"234,896,275.230",22,0.001,"5,130,439.150",0.022,4424,0.114,"82,205,026.900",0.350,335,0.009,"28,685,142.340",0.122,19876,0.510,"100,181,929.290",0.426,14287,0.367,"18,693,737.550",0.080
3,3_medio,17348,"161,197,608.530",13,0.001,"4,263,457.180",0.026,3913,0.226,"77,580,099.050",0.481,254,0.015,"20,626,727.080",0.128,9523,0.549,"53,553,579.580",0.332,3645,0.210,"5,173,745.640",0.032
4,4_grande,31255,"732,942,051.710",201,0.006,"114,515,571.730",0.156,13359,0.427,"328,691,074.040",0.448,2388,0.076,"206,663,615.150",0.282,12986,0.415,"80,044,107.450",0.109,2321,0.074,"3,027,683.340",0.004


In [ ]:
df_values_by_person_type_uf = territorial.aggregate_execution_by_person_type(df_cubo=df_cubo, by_filter='UF')
df_values_by_person_type_state = territorial.aggregate_execution_by_person_type(df_cubo=df_cubo, by_filter='ESTADO')
df_values_by_person_type_municipality = territorial.aggregate_execution_by_person_type(df_cubo=df_cubo, by_filter='MUNICIPIO')

# df_values_by_person_type_uf.to_csv(settings.DATA_PATH_SECTION2 / 'aggregate_execution_by_person_type_uf.csv')
# df_values_by_person_type_state.to_csv(settings.DATA_PATH_SECTION2 / 'aggregate_execution_by_person_type_state.csv')
# df_values_by_person_type_municipality.to_csv(settings.DATA_PATH_SECTION2 / 'aggregate_execution_by_person_type_municipality.csv')

In [27]:
df_box = territorial.make_boxplot_df_faixa_valor(df_aux=df_aux)

In [125]:
df_box

,visao,uf_bbagil,faixa_vlr_pago_ju_bbagil,metrica,valor_boxplot,unidade_observacao
0,ESTADO,AC,Até 2 mil,quantidade_contemplados,13.000,uf_faixa
1,ESTADO,AC,De 2 a 10 mil,quantidade_contemplados,25.000,uf_faixa
2,ESTADO,AC,De 10 a 50 mil,quantidade_contemplados,368.000,uf_faixa
3,ESTADO,AC,De 50 a 200 mil,quantidade_contemplados,36.000,uf_faixa
4,ESTADO,AC,Acima de 200 mil,quantidade_contemplados,10.000,uf_faixa
...,...,...,...,...,...,...
22157,ESTADO,TO,Acima de 200 mil,valor_transacao_total_bbagil,"367,000.000",contemplado
22158,ESTADO,TO,Acima de 200 mil,valor_transacao_total_bbagil,"237,000.000",contemplado
22159,ESTADO,TO,Acima de 200 mil,valor_transacao_total_bbagil,"237,000.000",contemplado
22160,ESTADO,TO,Acima de 200 mil,valor_transacao_total_bbagil,"237,000.000",contemplado


In [71]:
df_box_qtd_contemplados = df_box[df_box['metrica'] == 'quantidade_contemplados']
df_box_qtd_contemplados.to_csv(settings.DATA_PATH_SECTION2 / 'faixa_valor_box_plot_qtd_contemplados_state.csv')

In [159]:
import pandas as pd

# Filtra apenas ESTADO
df_estado = df_aux.copy()

# Garante que a coluna de valor está numérica
df_estado["valor_transacao_total_bbagil"] = pd.to_numeric(
    df_estado["valor_transacao_total_bbagil"],
    errors="coerce"
)

# Remove valores nulos
serie_valor = df_estado["valor_transacao_total_bbagil"].dropna()

# Limite superior para média aparada
p99 = serie_valor.quantile(0.99)

# Média aparada: remove os 1% maiores valores
media_aparada_1pct = serie_valor[serie_valor <= p99].mean()

# Tabela geral de percentis e quartis
df_percentis_estado = pd.DataFrame({
    "tipo_ente": ["GERAL"],
    "quantidade_contemplados": [serie_valor.count()],
    "valor_minimo": [serie_valor.min()],
    "p1": [serie_valor.quantile(0.01)],
    "q1": [serie_valor.quantile(0.25)],
    "q2_mediana": [serie_valor.quantile(0.50)],
    "q3": [serie_valor.quantile(0.75)],
    "p99": [p99],
    "valor_maximo": [serie_valor.max()],
    "media": [serie_valor.mean()],
    "media_aparada_1pct": [media_aparada_1pct],
    "desvio_padrao": [serie_valor.std()]
})

df_percentis_estado

,tipo_ente,quantidade_contemplados,valor_minimo,p1,q1,q2_mediana,q3,p99,valor_maximo,media,media_aparada_1pct,desvio_padrao
0,GERAL,166886,375.000,500.000,"2,000.000","4,950.740","12,500.000","200,000.000","22,109,764.920","17,053.531","12,855.080","106,478.885"


In [ ]:
df_box_qtd_contemplados = df_box[df_box['metrica'] == 'valor_t1ransacao_total_bbagil']
df_box_qtd_contemplados.to_csv(settings.DATA_PATH_SECTION2 / 'faixa_valor_box_plot_valor_transacao_total_bbagil_state.csv')

In [168]:
importlib.reload(territorial)

<module 'src.aggregations.territorial' from 'c:\\Users\\gabiru\\Documents\\GitHub\\pnab-data-vis\\src\\aggregations\\territorial.py'>

In [40]:
# resumo_faixa_valor_por_porte.csv
df_resumo_faixas_porte = territorial.resumo_faixa_valor_por_porte(df_cubo=df_cubo)
# df_resumo_faixas_porte.to_csv(settings.DATA_PATH_SECTION2 / 'faixa_valor_porte_populacional.csv')

In [41]:
df_resumo_faixas_porte_cut = df_resumo_faixas_porte[df_resumo_faixas_porte['porte_populacional'] != '-99']

In [67]:
df_territorios_especiais = territorial.resumo_territorios_especiais_por_uf(df_cubo=df_cubo)
df_territorios_especiais.to_csv(settings.DATA_PATH_SECTION2 / 'territorios_especiais_por_uf.csv')

In [63]:
df_territorios_especiais = territorial.resumo_territorios_especiais_por_uf(df_cubo=df_cubo, visao='ESTADO')
# df_territorios_especiais.to_csv(settings.DATA_PATH_SECTION2 / 'territorios_especiais_por_municipio.csv')

In [15]:
df_agg_faixas_regiao = territorial.aggregate_faixa_valor_ju_wide_by_regiao(df_cubo=df_cubo)

In [16]:
df_agg_faixas_regiao

,regiao,total_contemplados_uf,valor_total_uf,qtd_ate_2_mil,perc_qtd_ate_2_mil,valor_ate_2_mil,perc_valor_ate_2_mil,qtd_de_2_a_10_mil,perc_qtd_de_2_a_10_mil,valor_de_2_a_10_mil,perc_valor_de_2_a_10_mil,qtd_de_10_a_50_mil,perc_qtd_de_10_a_50_mil,valor_de_10_a_50_mil,perc_valor_de_10_a_50_mil,qtd_de_50_a_200_mil,perc_qtd_de_50_a_200_mil,valor_de_50_a_200_mil,perc_valor_de_50_a_200_mil,qtd_acima_de_200_mil,perc_qtd_acima_de_200_mil,valor_acima_de_200_mil,perc_valor_acima_de_200_mil
0,Norte,14504,"303,553,050.960",3109,0.214,"4,222,896.090",0.014,6412,0.442,"34,322,684.290",0.113,4139,0.285,"100,204,251.410",0.330,753,0.052,"70,470,381.050",0.232,91,0.006,"94,332,838.120",0.311
1,Nordeste,79446,"852,268,894.220",32985,0.415,"40,446,875.550",0.047,32083,0.404,"160,070,111.710",0.188,11608,0.146,"283,494,131.220",0.333,2474,0.031,"231,025,932.590",0.271,296,0.004,"137,231,843.150",0.161
2,Centro-Oeste,9335,"237,420,986.840",1846,0.198,"2,410,110.790",0.010,4313,0.462,"23,389,777.990",0.099,2172,0.233,"55,635,914.330",0.234,864,0.093,"83,029,309.050",0.350,140,0.015,"72,955,874.680",0.307
3,Sudeste,45655,"1,066,203,024.520",8784,0.192,"11,271,956.700",0.011,19547,0.428,"106,966,998.930",0.100,13694,0.300,"317,014,751.370",0.297,2983,0.065,"285,456,080.660",0.268,647,0.014,"345,493,236.860",0.324
4,Sul,17946,"386,549,637.450",2313,0.129,"2,899,060.020",0.007,8138,0.453,"47,068,043.270",0.122,6048,0.337,"139,969,570.820",0.362,1274,0.071,"128,648,301.770",0.333,173,0.010,"67,964,661.570",0.176


In [17]:
df_regioes = territorial.resumo_por_regiao(df_aux=df_aux)
# df_regioes.to_csv(settings.DATA_PATH_SECTION2 / 'resumo_por_regiao_uf.csv')

In [18]:
df_regioes

,nome_macrorregiao,valor_total_por_regiao,valor_medio_por_regiao,valor_media_aparada_1pct_por_regiao,valor_mediano_por_regiao,quantidade_contemplados_por_regiao,percentual_valor_por_regiao,percentual_quantidade_contemplados_por_regiao,visao
0,Norte,"303,553,050.960","20,928.920","13,900.745","6,000.000",14504,0.107,0.087,UF
1,Nordeste,"852,268,894.220","10,727.650","8,222.396","3,000.000",79446,0.299,0.476,UF
2,Centro-Oeste,"237,420,986.840","25,433.421","19,403.457","5,609.610",9335,0.083,0.056,UF
3,Sudeste,"1,066,203,024.520","23,353.478","17,994.297","6,900.000",45655,0.375,0.274,UF
4,Sul,"386,549,637.450","21,539.599","17,925.222","8,822.630",17946,0.136,0.108,UF


In [67]:
# tabela_resumo_estado_municipio.csv
df_resumo_est_mun = territorial.tabela_resumo_estado_municipio(df_aux=df_aux)
df_resumo_est_mun.to_csv(settings.DATA_PATH_SECTION2 / 'tabela_resumo_estado_municipio.csv')

In [178]:
df_resumo_est_mun

,indicador,Estados,Municípios
0,Número de contemplados,22.050,144.836
1,Ticket médio dos pagamentos,R$ 52.711,R$ 7.839
2,"Concentração dos contemplados, por faixa de valor",De 10 a 50 mil,De 2 a 10 mil
3,"Concentração do recurso executado, por faixa d...",Acima de 200 mil,De 10 a 50 mil


# Section 3

In [58]:
importlib.reload(person)

<module 'src.aggregations.person' from 'c:\\Users\\gabiru\\Documents\\GitHub\\pnab-data-vis\\src\\aggregations\\person.py'>

In [206]:
# aggregate_contemplados_pf_pj_proportion.csv
df_person = person.aggregate_contemplados_pf_pj_proportion(df_cubo=df_cubo)
df_person.to_csv(settings.DATA_PATH_SECTION3 / 'aggregate_contemplados_pf_pj_proportion.csv')

In [214]:
df_person_state = person.aggregate_contemplados_pf_pj_proportion(df_cubo=df_cubo, by_filter='ESTADO')
df_person_state.to_csv(settings.DATA_PATH_SECTION3 / 'aggregate_contemplados_pf_pj_proportion_state.csv')

In [215]:
df_person_state

,quantidade_contemplados,perc_quantidade_contemplados,quantidade_contemplados_pf,perc_quantidade_contemplados_pf,quantidade_contemplados_pj,perc_quantidade_contemplados_pj,valor_contemplados,perc_valor_contemplados,valor_contemplados_pf,perc_valor_contemplados_pf,valor_contemplados_pj,perc_valor_contemplados_pj,valor_medio_contemplados_pf,media_aparada_1pct_valor_pf,valor_medio_contemplados_pj,media_aparada_1pct_valor_pj
0,22050,1.000,14996,0.680,7054,0.320,"1,450,514,326.100",1.000,"512,696,665.520",0.353,"937,817,660.580",0.647,"34,188.895","32,198.427","132,948.350","116,228.895"


In [216]:
df_person_municipio = person.aggregate_contemplados_pf_pj_proportion(df_cubo=df_cubo, by_filter='MUNICIPIO')
df_person_municipio.to_csv(settings.DATA_PATH_SECTION3 / 'aggregate_contemplados_pf_pj_proportion_municipio.csv')

In [217]:
df_person_municipio

,quantidade_contemplados,perc_quantidade_contemplados,quantidade_contemplados_pf,perc_quantidade_contemplados_pf,quantidade_contemplados_pj,perc_quantidade_contemplados_pj,valor_contemplados,perc_valor_contemplados,valor_contemplados_pf,perc_valor_contemplados_pf,valor_contemplados_pj,perc_valor_contemplados_pj,valor_medio_contemplados_pf,media_aparada_1pct_valor_pf,valor_medio_contemplados_pj,media_aparada_1pct_valor_pj
0,144836,1.000,119610,0.826,25226,0.174,"1,395,481,267.890",1.000,"741,987,235.900",0.532,"653,494,031.990",0.468,"6,203.388","5,815.302","25,905.575","21,525.165"


In [ ]:
# aggregate_cnpj_mei_proportion.csv
df_person_mei = person.aggregate_cnpj_mei_proportion(df_cubo=df_cubo)
df_person_mei.to_csv(settings.DATA_PATH_SECTION3 / 'aggregate_cnpj_mei_proportion.csv')

In [212]:
df_person_mei

,quantidade_contemplados_cnpj,perc_quantidade_contemplados_cnpj,quantidade_contemplados_mei,perc_quantidade_contemplados_mei,quantidade_contemplados_nao_mei,perc_quantidade_contemplados_nao_mei,valor_contemplados_cnpj,perc_valor_contemplados_cnpj,valor_contemplados_mei,perc_valor_contemplados_mei,valor_contemplados_nao_mei,perc_valor_contemplados_nao_mei,valor_medio_contemplados_cnpj,valor_medio_contemplados_mei,valor_medio_contemplados_nao_mei,media_aparada_1pct_valor_cnpj,media_aparada_1pct_valor_mei,media_aparada_1pct_valor_nao_mei
0,32280,1.000,9076,0.281,23204,0.719,"1,591,311,692.570",1.000,"238,855,895.500",0.150,"1,352,455,797.070",0.850,"49,297.140","26,317.309","58,285.459","38,847.488","23,149.502","45,678.340"


In [196]:
df_cubo_cnpj = df_cubo[df_cubo['tipo_documento']=='CNPJ']

In [198]:
df_cubo_cnpj['valor_transacao'].sum()

np.float64(1591311692.5700002)

In [ ]:
df_cubo_mei = df_cubo[df_cubo['cnpj_optante_mei']==1]

In [195]:
df_cubo_cnpj['valor_transacao'].sum()

np.float64(238855895.5)

0.15010000656392022

In [8]:
# aggregate_contemplados_by_sexo_proportion.csv
df_sexo = person.aggregate_contemplados_by_sexo_proportion(df_cubo=df_cubo)
df_sexo.to_csv(settings.DATA_PATH_SECTION3 / 'aggregate_contemplados_by_sexo_proportion.csv')

In [50]:
df_sexo.head(2)

,quantidade_contemplados,perc_quantidade_contemplados,valor_contemplados,perc_valor_contemplados,quantidade_contemplados_feminino,perc_quantidade_contemplados_feminino,valor_contemplados_feminino,perc_valor_contemplados_feminino,quantidade_contemplados_masculino,perc_quantidade_contemplados_masculino,valor_contemplados_masculino,perc_valor_contemplados_masculino
0,134593,1,1254423238,1,62943,0.467654,578195818,0.460926,71650,0.532346,676227421,0.539074


In [223]:
# aggregate_valor_quantity_by_age_group_sexo_wide
df_age_group = person.aggregate_valor_quantity_by_age_group_sexo_wide(df_cubo=df_cubo)
df_age_group.to_csv(settings.DATA_PATH_SECTION3 / 'aggregate_valor_quantity_by_age_group_sexo_wide.csv')

In [224]:
df_age_group

sexo_tratado,faixa_etaria,valor_recebido_feminino,valor_recebido_masculino,valor_recebido_total,perc_valor_feminino_na_faixa,perc_valor_masculino_na_faixa,perc_valor_total_geral,quantidade_contemplados_feminino,quantidade_contemplados_masculino,quantidade_contemplados_total,perc_quantidade_feminino_na_faixa,perc_quantidade_masculino_na_faixa,perc_quantidade_total_geral
0,15-24 anos,36202863,37636414,73839277,0.490,0.510,0.059,4964,5844,10808,0.459,0.541,0.080
1,25-54 anos,419474838,516425879,935900716,0.448,0.552,0.746,42394,52559,94953,0.446,0.554,0.705
2,55-64 anos,75062775,70779785,145842560,0.515,0.485,0.116,9565,7653,17218,0.556,0.444,0.128
3,65+ anos,47455343,51385344,98840686,0.480,0.520,0.079,6020,5594,11614,0.518,0.482,0.086


In [ ]:
# aggregate_value_quantity_by_age_group_region_wide
df_age_region = person.aggregate_value_quantity_by_age_group_region_wide(df_cubo=df_cubo)
df_age_region.to_csv(settings.DATA_PATH_SECTION3 / 'aggregate_value_quantity_by_age_group_region_wide.csv')

In [221]:
# aggregate_sexo_uf_ibge_pnab.csv
df_resumo_sexo = person.aggregate_sexo_uf_ibge_pnab(df_cubo=df_cubo)
df_resumo_sexo.to_csv(settings.DATA_PATH_SECTION3 / 'aggregate_sexo_uf_ibge_pnab.csv')


In [227]:
df_cubo['naturezajuridica_agrupada_receita_cnpj'].unique()

<ArrowStringArray>
[               'Microempresa-ME',                            'MEI',
 'Empresa de Pequeno Porte (EPP)',          'Administração Pública',
  'Entidades sem fins lucrativos',         'Entidades Empresariais',
                              nan]
Length: 7, dtype: str

In [231]:
df_cnpj_natureza = person.aggregate_cnpj_natureza_juridica(df_cubo=df_cubo)
df_cnpj_natureza.to_csv(settings.DATA_PATH_SECTION3 / 'aggregate_cnpj_natureza_juridica.csv')

In [62]:
df_cnpj_natureza_regiao = person.aggregate_cnpj_natureza_juridica_por_regiao(df_cubo=df_cubo)
df_cnpj_natureza_regiao.to_csv(settings.DATA_PATH_SECTION3 / 'aggregate_cnpj_natureza_juridica_por_regiao.csv')

In [248]:
importlib.reload(person)

<module 'src.aggregations.person' from 'c:\\Users\\gabiru\\Documents\\GitHub\\pnab-data-vis\\src\\aggregations\\person.py'>

In [63]:
df_cnae_principal = person.top_cnaes_cnpj(df_cubo=df_cubo, apenas_cnae_cultural=False)
df_cnae_principal.to_csv(settings.DATA_PATH_SECTION3 / 'top_cnaes_cnpj.csv')

In [247]:
df_cnae_principal

,ranking_valor,cnae_principal,quantidade_contemplados,valor_transacao,perc_quantidade_contemplados,perc_valor_transacao,visao_cnae
0,1,Atividades de associações de defesa de direito...,4430,"266,009,595.440",0.137,0.167,CNAE geral
1,2,Atividades de organizações associativas ligada...,2736,"222,281,615.900",0.085,0.140,CNAE geral
2,3,"Serviços de organização de feiras, congressos,...",2419,"100,786,253.140",0.075,0.063,CNAE geral
3,4,Produção musical,2532,"83,379,775.690",0.078,0.052,CNAE geral
4,5,Produção teatral,1251,"81,944,578.600",0.039,0.051,CNAE geral
5,6,Ensino de arte e cultura não especificado ante...,2424,"76,208,049.920",0.075,0.048,CNAE geral
6,7,"Artes cênicas, espetáculos e atividades comple...",844,"75,254,228.470",0.026,0.047,CNAE geral
7,8,"Atividades de produção cinematográfica, de víd...",638,"60,918,084.220",0.020,0.038,CNAE geral
8,9,Construção de edifícios,209,"38,946,105.750",0.006,0.024,CNAE geral
9,10,Atividades associativas não especificadas ante...,635,"35,377,694.310",0.020,0.022,CNAE geral


In [249]:
df_cnae_principal_cultura = person.top_cnaes_cnpj(df_cubo=df_cubo, apenas_cnae_cultural=True)
df_cnae_principal_cultura.to_csv(settings.DATA_PATH_SECTION3 / 'top_cnaes_cnpj_cultura.csv')

In [250]:
df_cnae_principal_cultura

,ranking_valor,cnae_principal,quantidade_contemplados,valor_transacao,perc_quantidade_contemplados,perc_valor_transacao,visao_cnae,total_quantidade_todos_cnaes,total_valor_todos_cnaes
0,1,Atividades de organizações associativas ligada...,2736,"222,281,615.900",0.085,0.140,CNAE cultural,32280,"1,591,311,692.570"
1,2,Produção musical,2532,"83,379,775.690",0.078,0.052,CNAE cultural,32280,"1,591,311,692.570"
2,3,Produção teatral,1251,"81,944,578.600",0.039,0.051,CNAE cultural,32280,"1,591,311,692.570"
3,4,Ensino de arte e cultura não especificado ante...,2424,"76,208,049.920",0.075,0.048,CNAE cultural,32280,"1,591,311,692.570"
4,5,"Artes cênicas, espetáculos e atividades comple...",844,"75,254,228.470",0.026,0.047,CNAE cultural,32280,"1,591,311,692.570"
5,6,"Atividades de produção cinematográfica, de víd...",638,"60,918,084.220",0.020,0.038,CNAE cultural,32280,"1,591,311,692.570"
6,7,Serviços de arquitetura,99,"18,495,996.530",0.003,0.012,CNAE cultural,32280,"1,591,311,692.570"
7,8,Atividades de museus e de exploração de lugare...,82,"18,051,711.120",0.003,0.011,CNAE cultural,32280,"1,591,311,692.570"
8,9,"Ensino de artes cênicas, exceto dança",346,"16,472,144.850",0.011,0.010,CNAE cultural,32280,"1,591,311,692.570"
9,10,"Gestão de espaços para artes cênicas, espetácu...",66,"14,729,970.000",0.002,0.009,CNAE cultural,32280,"1,591,311,692.570"


# Section 4

In [34]:
importlib.reload(labor)
importlib.reload(person)


<module 'src.aggregations.person' from 'c:\\Users\\gabiru\\Documents\\GitHub\\pnab-data-vis\\src\\aggregations\\person.py'>

In [257]:
# aggregate_vinculo_formal_labor.csv
df_not_in_mercado = labor.aggregate_vinculo_formal_labor(df_cubo=df_cubo)
df_not_in_mercado.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor.csv')

In [258]:
df_not_in_mercado

,numero_contemplados_sem_vinculo_trabalho_formal,numero_contemplados_com_vinculo_trabalho_formal,numero_contemplados_total,percentual_contemplados_sem_vinculo_trabalho_formal,percentual_contemplados_com_vinculo_trabalho_formal,valor_pago_sem_vinculo_trabalho_formal,valor_pago_com_vinculo_trabalho_formal,valor_pago_total,percentual_valor_pago_sem_vinculo_trabalho_formal,percentual_valor_pago_com_vinculo_trabalho_formal
0,74696,59910,134606,0.555,0.445,"649,385,460.010","605,298,441.410","1,254,683,901.420",0.518,0.482


In [259]:
# aggregate_vinculo_formal_labor_by_uf.csv
df_not_in_mercado_by_uf = labor.aggregate_vinculo_formal_labor_by_uf(df_cubo=df_cubo)
# df_not_in_mercado_by_uf.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor_by_uf.csv')

In [35]:
# aggregate_vinculo_formal_labor_by_uf_ibge.csv
# percentual_numero_contemplados_com_vinculo_no_total_brasil
df_rais_ibge = person.aggregate_vinculo_formal_labor_by_uf(
    df_cubo=df_cubo
)

df_rais_ibge.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor_by_uf_ibge.csv')

TypeError: aggregate_vinculo_formal_labor_by_uf() missing 1 required positional argument: 'df_rais_uf'

In [31]:
df_rais_ibge

,uf,codigo_uf,unidade_da_federacao,populacao_ibge_2024,total_vinculos_a_rais_2024,percentual_populacao_com_vinculo_rais_2024,numero_contemplados_sem_vinculo_trabalho_formal,numero_contemplados_com_vinculo_trabalho_formal,numero_contemplados_total,percentual_contemplados_sem_vinculo_trabalho_formal,percentual_contemplados_com_vinculo_trabalho_formal,diferenca_pp_contemplados_com_vinculo_vs_populacao_uf,razao_contemplados_com_vinculo_vs_populacao_uf,percentual_numero_contemplados_no_total_geral,percentual_numero_contemplados_sem_vinculo_no_total_geral,percentual_numero_contemplados_com_vinculo_no_total_geral,valor_pago_sem_vinculo_trabalho_formal,valor_pago_com_vinculo_trabalho_formal,valor_pago_total,percentual_valor_pago_sem_vinculo_trabalho_formal,percentual_valor_pago_com_vinculo_trabalho_formal,percentual_valor_pago_no_total_geral,percentual_valor_pago_sem_vinculo_no_total_geral,percentual_valor_pago_com_vinculo_no_total_geral
0,AC,12,Acre,880631,180093,0.205,414.000,544.000,958.000,0.432,0.568,0.363,2.777,0.007,0.006,0.009,"6,029,226.360","8,393,534.760","14,422,761.120",0.418,0.582,0.011,0.009,0.014
1,AL,27,Alagoas,3220104,636259,0.198,"2,943.000","2,743.000","5,686.000",0.518,0.482,0.285,2.441,0.042,0.039,0.046,"15,522,910.730","17,571,027.770","33,093,938.500",0.469,0.531,0.026,0.024,0.029
2,AM,13,Amazonas,4281209,801349,0.187,830.000,875.000,"1,705.000",0.487,0.513,0.326,2.742,0.013,0.011,0.015,"22,207,375.410","20,924,331.920","43,131,707.330",0.515,0.485,0.034,0.034,0.035
3,AP,16,Amapá,802837,154926,0.193,228.000,297.000,525.000,0.434,0.566,0.373,2.932,0.004,0.003,0.005,"2,267,715.500","2,983,510.950","5,251,226.450",0.432,0.568,0.004,0.003,0.005
4,BA,29,Bahia,14850513,2774157,0.187,"8,964.000","6,749.000","15,713.000",0.570,0.430,0.243,2.299,0.117,0.120,0.113,"54,432,147.090","49,256,473.830","103,688,620.920",0.525,0.475,0.083,0.084,0.081
5,CE,23,Ceará,9233656,1871093,0.203,"4,271.000","3,147.000","7,418.000",0.576,0.424,0.222,2.094,0.055,0.057,0.053,"38,868,061.770","36,801,043.890","75,669,105.660",0.514,0.486,0.060,0.060,0.061
6,DF,53,Distrito Federal,2982818,1707305,0.572,174.000,121.000,295.000,0.590,0.410,-0.162,0.717,0.002,0.002,0.002,"5,995,132.440","3,743,567.140","9,738,699.580",0.616,0.384,0.008,0.009,0.006
7,ES,32,Espírito Santo,4102129,1101019,0.268,418.000,508.000,926.000,0.451,0.549,0.280,2.044,0.007,0.006,0.008,"6,648,941.860","7,627,382.160","14,276,324.020",0.466,0.534,0.011,0.010,0.013
8,GO,52,Goiás,7350483,1950312,0.265,"1,828.000","1,701.000","3,529.000",0.518,0.482,0.217,1.817,0.026,0.024,0.028,"33,130,113.390","30,021,027.510","63,151,140.900",0.525,0.475,0.050,0.051,0.050
9,MA,21,Maranhão,7010960,977685,0.139,"4,551.000","2,913.000","7,464.000",0.610,0.390,0.251,2.799,0.055,0.061,0.049,"23,193,784.490","21,275,349.840","44,469,134.330",0.522,0.478,0.035,0.036,0.035


In [37]:
# aggregate_vinculo_formal_labor_by_region.csv
df_not_in_mercado_by_region = labor.aggregate_vinculo_formal_labor_by_region(df_cubo=df_cubo)
df_not_in_mercado_by_region.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor_by_region.csv')


In [44]:
# aggregate_vinculo_formal_labor_by_sexo.csv
df_not_in_mercado_by_sexo = labor.aggregate_vinculo_formal_labor_by_sexo(df_cubo=df_cubo)
df_not_in_mercado_by_sexo.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor_by_sexo.csv')

In [50]:
# aggregate_vinculo_formal_labor_by_age_group.csv
df_not_in_mercado_by_age_group = labor.aggregate_vinculo_formal_labor_by_age_group(df_cubo=df_cubo)
df_not_in_mercado_by_age_group.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor_by_age_group.csv')

In [15]:
# aggregate_vinculo_formal_labor_by_raca_cor.csv
df_not_in_mercado_by_raca_cor = labor.aggregate_vinculo_formal_labor_by_raca_cor(df_cubo=df_cubo)
# df_not_in_mercado_by_raca_cor.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor_by_raca_cor.csv')


In [36]:
# resumo_raca_cor_com_vinculo_rais.csv
df_raca_cor_att = labor.resumo_raca_cor_com_vinculo_rais(df_cubo=df_cubo)
df_raca_cor_att.to_csv(settings.DATA_PATH_SECTION4 / 'resumo_raca_cor_com_vinculo_rais.csv')


In [ ]:
# resumo_escolaridade_com_vinculo_rais.csv
df_escolaridade = labor.resumo_escolaridade_com_vinculo_rais(df_cubo=df_cubo)
df_escolaridade.to_csv(settings.DATA_PATH_SECTION4 / 'resumo_escolaridade_com_vinculo_rais.csv')

In [45]:
df_escolaridade[['escolaridade_agregado_rais', 'valor_medio_transacao_com_vinculo']]

,escolaridade_agregado_rais,valor_medio_transacao_com_vinculo
0,Sem instrução e fundamental incompleto,"5,695.64"
1,Fundamental completo e médio incompleto,"6,635.20"
2,Médio completo e superior incompleto,"8,330.71"
3,Superior completo,"12,195.18"
4,Mestrado ou doutorado completo,"18,507.30"


In [10]:
df_cubo[df_cubo['raca_cor_desc_description'] == 'Parda – para a pessoa que se enquadrar como parda ou se declarar como mulata, cabocla, cafuza, mameluca ou mestiça de preto com pessoa de outra cor ou raça.']['quantidade'].sum()

np.int64(27071)

In [ ]:
# aggregate_raca_cor_vinculo_formal_labor_by_sexo
df_not_in_mercado_by_raca_cor_sexo = labor.aggregate_raca_cor_vinculo_formal_labor_by_sexo(df_cubo=df_cubo)
df_not_in_mercado_by_raca_cor_sexo.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_raca_cor_vinculo_formal_labor_by_sexo.csv')

In [54]:
# aggregate_vinculo_formal_labor_by_escolaridade.csv
df_not_in_mercado_escolaridade = labor.aggregate_vinculo_formal_labor_by_escolaridade(df_cubo=df_cubo)
# df_not_in_mercado_escolaridade.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor_by_escolaridade.csv')

In [55]:
df_not_in_mercado_escolaridade

,escolaridade_agregado_rais,numero_contemplados_sem_vinculo_trabalho_formal,numero_contemplados_com_vinculo_trabalho_formal,numero_contemplados_total,percentual_contemplados_sem_vinculo_trabalho_formal,percentual_contemplados_com_vinculo_trabalho_formal,percentual_numero_contemplados_no_total_geral,percentual_numero_contemplados_sem_vinculo_no_total_geral,percentual_numero_contemplados_com_vinculo_no_total_geral,valor_pago_sem_vinculo_trabalho_formal,valor_pago_com_vinculo_trabalho_formal,valor_pago_total,percentual_valor_pago_sem_vinculo_trabalho_formal,percentual_valor_pago_com_vinculo_trabalho_formal,percentual_valor_pago_no_total_geral,percentual_valor_pago_sem_vinculo_no_total_geral,percentual_valor_pago_com_vinculo_no_total_geral
0,Sem informação,"74,426.00",0.00,"74,426.00",1.00,0.00,0.55,1.00,0.00,"646,871,967.68",0.00,"646,871,967.68",1.00,0.00,0.52,1.00,0.00
1,Médio completo e superior incompleto,0.00,"26,539.00","26,539.00",0.00,1.00,0.20,0.00,0.44,0.00,"221,132,646.28","221,132,646.28",0.00,1.00,0.18,0.00,0.36
2,Superior completo,0.00,"25,772.00","25,772.00",0.00,1.00,0.19,0.00,0.43,0.00,"313,612,032.62","313,612,032.62",0.00,1.00,0.25,0.00,0.52
3,Fundamental completo e médio incompleto,0.00,"3,716.00","3,716.00",0.00,1.00,0.03,0.00,0.06,0.00,"24,640,999.19","24,640,999.19",0.00,1.00,0.02,0.00,0.04
4,Sem instrução e fundamental incompleto,0.00,"2,224.00","2,224.00",0.00,1.00,0.02,0.00,0.04,0.00,"12,697,694.06","12,697,694.06",0.00,1.00,0.01,0.00,0.02
5,Mestrado ou doutorado completo,0.00,"1,929.00","1,929.00",0.00,1.00,0.01,0.00,0.03,0.00,"35,728,561.59","35,728,561.59",0.00,1.00,0.03,0.00,0.06


In [69]:
# aggregate_vinculo_trabalho_formal_by_escolaridade_clean.csv
df_not_in_mercado_escolaridade_clean = labor.aggregate_vinculo_trabalho_formal_by_escolaridade_sem_sem_informacao(df_cubo=df_cubo)
df_not_in_mercado_escolaridade_clean.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_trabalho_formal_by_escolaridade_clean.csv')

In [ ]:
# CBOS

np.float64(12415168.75)

In [236]:
# aggregate_cbo_rais.csv
df_cbo_rais = labor.aggregate_cbo_rais(df_cubo=df_cubo)
# df_cbo_rais.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_cbo_rais.csv')

In [237]:
df_cubo

,ente,tipo_ente,tipo_documento,faixa_vlr_pago,uf,nome_ente,regiao,flag_capital,porte_populacional,Sexo,Estrangeiro,NomeNaturezaOcupacao,NomeOcupacaoPrincipal,faixa_etaria,flag_cpf_mei,cnaePrincipal,naturezaJuridica,porte,cnpj_optante_mei,raca_cor_desc_description,escolaridade_description,ind_deficiencia,tipo_deficiencia_description,tipo_vinculo_description,faixa_salarial_rais,CBO_2002_RAIS,cbo_codigo,cbo_descricao,SITUACAO,cod_situacao_nome,cod_tipo_nome,pessoaCad_cadunico,familiaPBF_cadunico,fxRendaFamiliarTotal_desc_cadunico,caracDomicilio_desc_cadunico,fxRendaPerCapita_desc_cadunico,pertence_bpc,categoria_municipio_ibge,faixa_vlr_pago_ju_bbagil,situacao_renda_cadunico,cod_cnae_principal_receita_cnpj,cnaesecundarias_receita_cnpj,descr_cnae_principal_receita_cnpj,naturezajuridica_agrupada_receita_cnpj,flag_cnae_cultural,tipo_vinculo_agregado_rais,escolaridade_agregado_rais,flag_cbo_cultural_rais,flag_join_rais,flag_cnae_educacao_receita_cnpj,flag_cnae_audiovisual_receita_cnpj,local_residencia_contemplados,quantidade,valor_transacao,min_valor_transacao,max_valor_transacao,sum_populacao
0,AC_Acre_12,ESTADO,CNPJ,1 milhão a 10 milhões,AC,Acre,Norte,False,-99,NaN,NaN,NaN,NaN,NaN,False,"{'codigo': '7820500', 'descricao': 'Locação de...","{'codigo': '2062', 'descricao': 'Sociedade Emp...",01 - MicroEmpresa-ME,0.000,NaN,NaN,NaN,NaN,NaN,NÃO SE APLICA,NaN,NaN,NaN,Urbana,Área urbana de alta densidade de edificações d...,Não especial,NaN,NaN,NaN,NaN,NaN,NaN,Capital,Acima de 200 mil,NaN,"7,820,500.000","[{'codigo': '4120400', 'descricao': 'Construçã...",Locação de mão-de-obra temporária,Microempresa-ME,CNAE NAO CULTURAL,NaN,NaN,NaN,False,CNAE NAO EDUCACAO,CNAE NAO AUDIOVISUAL,Capital,1,"2,034,002.650","2,034,002.650","2,034,002.650",880631
1,AC_Acre_12,ESTADO,CNPJ,10 a 50 mil,AC,Acre,Norte,False,-99,NaN,NaN,NaN,NaN,NaN,False,"{'codigo': '1629301', 'descricao': 'Fabricação...","{'codigo': '2135', 'descricao': 'Empresário (I...",01 - MicroEmpresa-ME,1.000,NaN,NaN,NaN,NaN,NaN,NÃO SE APLICA,NaN,NaN,NaN,Urbana,Área urbana de alta densidade de edificações d...,Não especial,NaN,NaN,NaN,NaN,NaN,NaN,Interior,De 10 a 50 mil,NaN,"1,629,301.000","[{'codigo': '4330404', 'descricao': 'Serviços ...","Fabricação de artefatos diversos de madeira, e...",MEI,CNAE NAO CULTURAL,NaN,NaN,NaN,False,CNAE NAO EDUCACAO,CNAE NAO AUDIOVISUAL,Interior,1,"50,000.000","50,000.000","50,000.000",880631
2,AC_Acre_12,ESTADO,CNPJ,10 a 50 mil,AC,Acre,Norte,False,-99,NaN,NaN,NaN,NaN,NaN,False,"{'codigo': '5911101', 'descricao': 'Estúdios c...","{'codigo': '2062', 'descricao': 'Sociedade Emp...",01 - MicroEmpresa-ME,0.000,NaN,NaN,NaN,NaN,NaN,NÃO SE APLICA,NaN,NaN,NaN,Urbana,Área urbana de alta densidade de edificações d...,Não especial,NaN,NaN,NaN,NaN,NaN,NaN,Capital,De 10 a 50 mil,NaN,"5,911,101.000","[{'codigo': '1811302', 'descricao': 'Impressão...",Estúdios cinematográficos,Microempresa-ME,CNAE CULTURAL,NaN,NaN,NaN,False,CNAE NAO EDUCACAO,CNAE AUDIOVISUAL,Capital,1,"18,000.000","18,000.000","18,000.000",880631
3,AC_Acre_12,ESTADO,CNPJ,10 a 50 mil,AC,Acre,Norte,False,-99,NaN,NaN,NaN,NaN,NaN,False,"{'codigo': '5911101', 'descricao': 'Estúdios c...","{'codigo': '2135', 'descricao': 'Empresário (I...",01 - MicroEmpresa-ME,0.000,NaN,NaN,NaN,NaN,NaN,NÃO SE APLICA,NaN,NaN,NaN,Urbana,Área urbana de alta densidade de edificações d...,Não especial,NaN,NaN,NaN,NaN,NaN,NaN,Capital,De 10 a 50 mil,NaN,"5,911,101.000","[{'codigo': '1813001', 'descricao': 'Impressão...",Estúdios cinematográficos,Microempresa-ME,CNAE CULTURAL,NaN,NaN,NaN,False,CNAE NAO EDUCACAO,CNAE AUDIOVISUAL,Capital,1,"18,000.000","18,000.000","18,000.000",880631
4,AC_Acre_12,ESTADO,CNPJ,10 a 50 mil,AC,Acre,Norte,False,-99,NaN,NaN,NaN,NaN,NaN,False,"{'codigo': '5911199', 'descricao': 'Atividades...","{'codigo': '2062', 'descricao': 'Sociedade Emp...",01 - MicroEmpresa-ME,0.000,NaN,NaN,NaN,NaN,NaN,NÃO SE APLICA,NaN,NaN,NaN,Urbana,Área urbana de alta densidade de edificações d...,Não especial,NaN,NaN,NaN,NaN,NaN,NaN,Capita

# Section 5

In [19]:
importlib.reload(cadunico)

<module 'src.aggregations.cadunico' from 'c:\\Users\\gabiru\\Documents\\GitHub\\pnab-data-vis\\src\\aggregations\\cadunico.py'>

In [47]:
df_cad_unico = pd.read_parquet(settings.DATA_PATH / 'input_data' / 'non-public' /'dim_cadunico__2026-05-19_18-17.parquet')

In [22]:
# aggregate_cadunico_summary.csv
df_cubo_cadunico = cadunico.aggregate_cadunico_summary(df_cubo=df_cubo)
df_cubo_cadunico.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_summary.csv')

In [21]:
df_cubo_cadunico

,perc_contemplados_cadunico,qtd_contemplados_cadunico,qtd_documentos_unicos_cadunico,valor_recebido_cadunico,perc_valor_cadunico,media_valor_recebido_cadunico,media_aparada_valor_recebido_cadunico
0,0.434,58407,57338,"365,789,773.530",0.292,"6,262.773","5,361.884"


In [13]:
# aggregate_cadunico_profile_summary.csv
df_cubo_cadunico_sexo_idade = cadunico.aggregate_cadunico_profile_summary(df_cubo=df_cubo)
sexo = df_cubo_cadunico_sexo_idade[df_cubo_cadunico_sexo_idade['dimensao'] == 'Sexo']
faixa_etaria = df_cubo_cadunico_sexo_idade[df_cubo_cadunico_sexo_idade['dimensao'] == 'Faixa etária']
sexo.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_profile_summary_by_sexo.csv')
faixa_etaria.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_profile_summary_by_faixa_etaria.csv')


In [ ]:
# aggregate_cadunico_faixa_etaria_by_sexo.csv
df_cubo_cadunico_sexo_idade_juntos = cadunico.aggregate_cadunico_faixa_etaria_by_sexo(df_cubo=df_cubo)
df_cubo_cadunico_sexo_idade_juntos.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_faixa_etaria_by_sexo.csv')

In [ ]:
# aggregate_cadunico_by_situacao_renda.csv
df_cad_unico_situacao_renda = cadunico.aggregate_cadunico_by_situacao_renda(df_cubo=df_cubo)
df_cad_unico_situacao_renda.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_by_situacao_renda.csv')


In [ ]:
# aggregate_cadunico_by_fx_renda_per_capita.csv
df_cad_unico_situacao_faixa_renda_percapita = cadunico.aggregate_cadunico_by_fx_renda_per_capita(df_cubo=df_cubo)
df_cad_unico_situacao_faixa_renda_percapita.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_by_fx_renda_per_capita.csv')

In [ ]:
# aggregate_cadunico_by_situacao_domicilio.csv
df_cad_unico_domicilio_situacao = cadunico.aggregate_cadunico_by_situacao_domicilio(df_cubo=df_cubo)
df_cad_unico_domicilio_situacao.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_by_situacao_domicilio.csv')

In [187]:
# aggregate_cadunico_by_population_size.csv
df_cad_unico_porte_populacional = cadunico.aggregate_cadunico_by_population_size(df_cubo=df_cubo)
df_cad_unico_porte_populacional.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_by_population_size.csv')


In [6]:
# aggregate_cadunico_by_uf.csv
df_cad_unico_by_uf = cadunico.aggregate_cadunico_by_uf(df_cubo=df_cubo)
df_cad_unico_by_uf.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_by_uf.csv')

In [4]:
8629/15713

0.5491631133456374

In [7]:
df_cad_unico_by_uf

,uf,qtd_contemplados_cadunico,qtd_contemplados_total_uf,perc_qtd_cadunico_na_uf,perc_qtd_cadunico_brasil,perc_qtd_total_brasil,valor_contemplados_cadunico,valor_contemplados_total_uf,perc_valor_cadunico_brasil,perc_valor_total_brasil
0,BA,8629,15713,0.549,0.148,0.117,"42,920,208.140","103,688,620.920",0.117,0.083
1,PE,6224,12062,0.516,0.107,0.090,"34,869,304.640","92,429,310.590",0.095,0.074
2,MG,5790,17290,0.335,0.099,0.128,"40,741,509.630","162,053,698.710",0.111,0.129
3,PB,5292,9278,0.570,0.091,0.069,"18,514,005.580","44,521,802.910",0.051,0.035
4,CE,3935,7418,0.530,0.067,0.055,"28,699,794.430","75,669,105.660",0.078,0.060
5,MA,3893,7464,0.522,0.067,0.055,"17,397,799.210","44,469,134.330",0.048,0.035
6,PA,3471,6850,0.507,0.059,0.051,"20,955,312.970","51,903,775.130",0.057,0.041
7,PI,3187,5294,0.602,0.055,0.039,"8,123,633.390","18,821,684.590",0.022,0.015
8,AL,2778,5686,0.489,0.048,0.042,"11,446,161.390","33,093,938.500",0.031,0.026
9,RN,2774,4787,0.579,0.047,0.036,"12,845,543.630","28,243,213.700",0.035,0.023


In [12]:
# aggregate_cadunico_representacao_by_uf
df_cad_percs = cadunico.aggregate_cadunico_representacao_by_uf(df_cubo=df_cubo)
df_cad_percs.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_representacao_by_uf.csv')


In [172]:
# aggregate_cadunico_by_value_group.csv
df_cad_unic_faixa_valor = cadunico.aggregate_cadunico_by_value_group(df_cubo=df_cubo)
df_cad_unic_faixa_valor.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_by_value_group.csv')


In [173]:
# aggregate_bolsa_familia_summary.csv
df_cad_unico_bpf = cadunico.aggregate_bolsa_familia_summary(df_cubo=df_cubo)
df_cad_unico_bpf.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_bolsa_familia_summary.csv')

In [180]:
# aggregate_bpc_summary.csv
df_cad_unico_bpc = cadunico.aggregate_bpc_summary(df_cubo=df_cubo)
df_cad_unico_bpc.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_bpc_summary.csv')

In [181]:
df_cpf_receita = pd.read_parquet(settings.DATA_PATH / 'input_data' / 'non-public' /'dim_cpf_receita__2026_05-19-13_09.parquet')

In [183]:
df_cpf_receita[df_cpf_receita['sexo_receita_cpf'] == 'Feminino']['cpf_receita_cpf'].nunique()

61026

In [184]:
df_cpf_receita[df_cpf_receita['sexo_receita_cpf'] == 'Masculino']['cpf_receita_cpf'].nunique()

68615

In [186]:
61026/(61026+68615)

0.47073071019199175